In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("github-token")

github_token = secret_value_0

!git clone https://{github_token}@github.com/manaal-m/acdc-cardiac-mri-segmentation.git
%cd acdc-cardiac-mri-segmentation

Cloning into 'acdc-cardiac-mri-segmentation'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 29 (delta 1), reused 25 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 289.71 KiB | 18.11 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/kaggle/working/acdc-cardiac-mri-segmentation


In [2]:
%pip install -q lightning nibabel segmentation-models-pytorch kagglehub torchinfo

import os
import json
import torch
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

from src.dataset import ACDC2DDataModule
from src.model import LitUNet2D
from src.benchmark import measure_batch_latency, model_size_mb, load_results, save_results, update_result

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.6 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but yo

In [3]:
print("Imports succeeded.")

Imports succeeded.


In [4]:
import kagglehub

data_path = kagglehub.dataset_download("samdazel/automated-cardiac-diagnosis-challenge-miccai17")
acdc_root = f"{data_path}/database/training"

print("Data downloaded to:", data_path)

Data downloaded to: /kaggle/input/datasets/samdazel/automated-cardiac-diagnosis-challenge-miccai17


In [5]:
# Config
ENCODERS = ["resnet18", "vgg16", "mobilenet_v2", "resnet50"]
MAX_EPOCHS = 50
BATCH_SIZE = 8
LR = 1e-3
SEEDS = [11]
all_seed_results = {}

CHECKPOINT_DIR = "checkpoints"
RESULTS_PATH = "results/all_results.json"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs("results", exist_ok=True)

In [6]:
import numpy as np

for seed in SEEDS:
    pl.seed_everything(seed, workers=True)
    results = load_results(RESULTS_PATH)

    dm = ACDC2DDataModule(acdc_root, batch_size=BATCH_SIZE, num_workers=2, seed=seed)
    dm.setup()

    print(f"\n{'='*40}")
    print(f"SEED {seed}")
    print(f"{'='*40}")
    seed_results = {}

    for encoder in ENCODERS:
        print(f"\nTraining: {encoder}")

        model = LitUNet2D(lr=LR, encoder_name=encoder)

        early_stop = EarlyStopping(monitor="val_dice", mode="max", patience=20, verbose=True)
        checkpoint = ModelCheckpoint(
            monitor="val_dice", mode="max", save_top_k=1,
            dirpath=CHECKPOINT_DIR,
            filename=f"seed{seed}_{encoder}-acdc-{{epoch:02d}}-{{val_dice:.4f}}",
        )
        trainer = Trainer(
            max_epochs=MAX_EPOCHS, callbacks=[early_stop, checkpoint],
            log_every_n_steps=5, enable_progress_bar=True, logger=False,
        )

        trainer.fit(model, dm)

        best_model = LitUNet2D.load_from_checkpoint(checkpoint.best_model_path)
        best_model = best_model.to("cuda")

        final_ckpt_path = os.path.join(CHECKPOINT_DIR, f"seed{seed}_{encoder}_best.pth")
        torch.save(best_model.state_dict(), final_ckpt_path)

        best_model.eval()
        batch_latency_ms = measure_batch_latency(best_model, dm.val_dataloader(), device="cuda", n_batches=50)

        test_res = trainer.test(best_model, datamodule=dm, verbose=False)
        size_mb = model_size_mb(best_model)

        seed_results[encoder] = {
            "dice": round(test_res[0].get("test_dice", -1), 4),
            "iou":  round(test_res[0].get("test_miou", -1), 4),
        }

        update_result(
            results, encoder,
            dice=seed_results[encoder]["dice"],
            iou=seed_results[encoder]["iou"],
            inference_ms_batch8_mean=round(batch_latency_ms, 2),
            model_size_mb=size_mb,
        )
        save_results(results, RESULTS_PATH)
        print(json.dumps(results[encoder], indent=2))

    all_seed_results[seed] = seed_results

print("\nALL TRAINING RUNS COMPLETE")

print(f"\n{'Encoder':<18} {'Dice mean':>12} {'Dice std':>10} {'mIoU mean':>12} {'mIoU std':>10}")
print("-" * 64)
for encoder in ENCODERS:
    dices = [all_seed_results[s][encoder]["dice"] for s in SEEDS]
    ious  = [all_seed_results[s][encoder]["iou"]  for s in SEEDS]
    print(f"{encoder:<18} {np.mean(dices):>12.4f} {np.std(dices):>10.4f} {np.mean(ious):>12.4f} {np.std(ious):>10.4f}")

Seed set to 11



SEED 11

Training: resnet18


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /kaggle/working/acdc-cardiac-mri-segmentation/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


...

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

Metric val_dice improved. New best score: 0.677
Metric val_dice improved by 0.108 >= min_delta = 0.0. New best score: 0.785
Metric val_dice improved by 0.076 >= min_delta = 0.0. New best score: 0.861
Metric val_dice improved by 0.022 >= min_delta = 0.0. New best score: 0.883
Metric val_dice improved by 0.006 >= min_delta = 0.0. New best score: 0.889
Metric val_dice improved by 0.014 >= min_delta = 0.0. New best score: 0.903
Metric val_dice improved by 0.006 >= min_delta = 0.0. New best score: 0.908
Metric val_dice improved by 0.001 >= min_delta = 0.0. New best score: 0.909
Metric val_dice improved by 0.001 >= min_delta = 0.0. New best score: 0.911
Metric val_dice improved by 0.003 >= min_delta = 0.0. New best score: 0.914
Metric val_dice improved by 0.001 >= min_delta = 0.0. New best score: 0.914
Metric val_dice improved by 0.001 >= min_delta = 0.0. New best score: 0.915
Metric val_dice improved by 0.001 >= min_delta = 0.0. New best score: 0.916
Metric val_dice improved by 0.002 >= min

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

{
  "encoder": "resnet50",
  "dice": 0.9021,
  "iou": 0.8718,
  "model_size_mb": 124.38,
  "inference_ms_batch8_mean": 206.78,
  "inference_ms_per_slice_mean": 25.5,
  "inference_ms_per_slice_std": 0.39
}

ALL TRAINING RUNS COMPLETE

Encoder               Dice mean   Dice std    mIoU mean   mIoU std
----------------------------------------------------------------
resnet18                 0.8948     0.0000       0.8633     0.0000
vgg16                    0.9084     0.0000       0.8783     0.0000
mobilenet_v2             0.9027     0.0000       0.8722     0.0000
resnet50                 0.9021     0.0000       0.8718     0.0000


In [7]:
CLASS_NAMES = {1: "LV", 2: "RV", 3: "Myocardium"}
per_class_all_seeds = {enc: {name: [] for name in CLASS_NAMES.values()} for enc in ENCODERS}

for seed in SEEDS:
    pl.seed_everything(seed, workers=True)
    dm = ACDC2DDataModule(acdc_root, batch_size=1, num_workers=2, seed=seed)
    dm.setup()
    test_loader = dm.test_dataloader()

    for encoder in ENCODERS:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"seed{seed}_{encoder}_best.pth")
        model = LitUNet2D(lr=LR, encoder_name=encoder)
        model.load_state_dict(torch.load(ckpt_path, map_location="cuda"))
        model = model.to("cuda").eval()

        all_preds, all_targets = [], []
        with torch.no_grad():
            for x, y in test_loader:
                pred = torch.argmax(model(x.to("cuda")), dim=1).squeeze().cpu()
                all_preds.append(pred)
                all_targets.append(y.squeeze())

        preds = torch.stack(all_preds)
        targets = torch.stack(all_targets)

        for cls, name in CLASS_NAMES.items():
            pred_bin = (preds == cls).float()
            tgt_bin  = (targets == cls).float()
            intersection = (pred_bin * tgt_bin).sum()
            d = (2 * intersection) / (pred_bin.sum() + tgt_bin.sum() + 1e-5)
            per_class_all_seeds[encoder][name].append(d.item())

print(f"{'Encoder':<18} {'LV':>16} {'RV':>16} {'Myocardium':>16}")
print("-" * 68)
for encoder in ENCODERS:
    row = f"{encoder:<18}"
    for name in CLASS_NAMES.values():
        vals = per_class_all_seeds[encoder][name]
        row += f"  {np.mean(vals):.4f}±{np.std(vals):.4f}"
    print(row)

Seed set to 11


Encoder                          LV               RV       Myocardium
--------------------------------------------------------------------
resnet18            0.8922±0.0000  0.8814±0.0000  0.9547±0.0000
vgg16               0.9059±0.0000  0.8781±0.0000  0.9585±0.0000
mobilenet_v2        0.9035±0.0000  0.8781±0.0000  0.9554±0.0000
resnet50            0.8988±0.0000  0.8837±0.0000  0.9565±0.0000


In [8]:
from torchinfo import summary

dummy = torch.zeros(1, 1, 512, 512).to("cuda")

print(f"{'Encoder':<18} {'Params (M)':>12} {'GMACs':>10}")
print("-" * 42)

for encoder in ENCODERS:
    model = LitUNet2D(lr=LR, encoder_name=encoder).to("cuda")
    s = summary(model, input_data=dummy, verbose=0)
    params = s.total_params / 1e6
    gmacs = s.total_mult_adds / 1e9
    print(f"{encoder:<18} {params:>12.2f} {gmacs:>10.2f}")

Encoder              Params (M)      GMACs
------------------------------------------
resnet18                  14.32      21.30
vgg16                     23.75      98.82
mobilenet_v2               6.63      13.46
resnet50                  32.52      42.23


In [11]:
import json
with open("seed11_results_backup.json", "w") as f:
    json.dump(all_seed_results, f, indent=2)
print("Backup saved")

Backup saved
